<a href="https://colab.research.google.com/github/TienNguyen0712/mimic_iv/blob/main/notebooks/00_mimic_iv_overviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Giới thiệu bộ dữ liệu MIMIC-IV**

**Khái niệm**

Medical Information Mart for Intensive Care (MIMIC) là tiêu chuẩn vàng và là một trong những kho tài nguyên quý giá nhất hiện nay dành cho nghiên cứu học máy trong y tế

Là một cơ sở dữ liệu lớn, ẩn danh, bao gồm thông tin sức khỏe của hàng chục nghìn bệnh nhân nội trú tại Trung tâm Y tế Beth Israel Deaconess (Boston, Hoa Kỳ) trong giai đoạn từ 2008 đến 2019.

Điểm khác biệt lớn nhất của MIMIC-IV so với các bộ dữ liệu y tế khác là tính đa chiều và chi tiết theo thời gian thực (longitudinal data), cho phép chúng ta theo dõi toàn bộ hành trình của bệnh nhân từ lúc nhập viện, điều trị tại khoa hồi sức tích cực (ICU) cho đến khi xuất viện.

**Cấu trúc**

Bộ dữ liệu được tổ chức theo mô hình quan hệ, chia thành các mô-đun chính:

- **Hosp (Hospital):** Chứa **thông tin tổng quát về hành trình của bệnh nhân** trong bệnh viện bao gồm: hồ sơ nhập viện, mã chẩn đoán (ICD-9/ICD-10), kết quả xét nghiệm phòng thí nghiệm (lab tests), và danh mục thuốc được kê đơn.

- **ICU:** Tập trung vào dữ liệu chi tiết khi bệnh nhân nằm tại **khoa hồi sức tích cực**. Đây là nơi chứa các thông tin nhạy cảm về thời gian như dấu hiệu sinh tồn (nhịp tim, huyết áp), các thủ thuật xâm lấn (thở máy, lọc máu) và liều lượng thuốc vận mạch.

In [ ]:
# Đọc dữ liệu từ drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install tabulate polars pyarrow duckdb

In [ ]:
import glob # Dọc đuôi file
import pandas as pd # Tương tác file
import gzip # Giải nén file
import duckdb # Chuyển và đọc dữ liệu nhanh
import polars as pl # Chuyển và đọc dữ liệu nhanh
import gc # Giải phóng bộ nhớ
import os # Thực hiện với phần tương tác
from tabulate import tabulate # Trình bày file

In [ ]:
def check_file(path):
  files = glob.glob(path, recursive=True) # Tìm số file có trong bộ dữ liệu
  print("Số file:", len(files))

  for f in files[:20]:
    size_mb = os.path.getsize(f) / (1024 ** 2)
    print(f)
    print(f"Size: {size_mb:.2f} MB")
    print("-" * 40)
  return files

def read_file(files):
  for f in files:
    try:
        # Đọc 1 dòng để lấy số cột (nhanh)
        df_head = pd.read_csv(f, compression='gzip', nrows=1)
        num_cols = df_head.shape[1]
        cols_name = list(df_head.columns)

        # Đếm tổng số dòng (không load vào RAM)
        with gzip.open(f, 'rb') as i_f:
            num_rows = sum(1 for line in i_f) - 1 # Trừ 1 cho dòng header

        print(f"File: {f}")
        print(f" - Kích thước: {num_rows} dòng x {num_cols} cột")
        print(f" - Các cột: {cols_name}\n")
        print("-" * 30)
    except Exception as e:
        print(f"Lỗi khi đọc file {f}: {e}")

## **1.1. Cấu trúc của các Module**

### **hosp**

In [ ]:
hosp = check_file('/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/hosp/*.csv.gz')

Số file: 22
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/hosp/prescriptions.csv.gz
Size: 578.21 MB
----------------------------------------
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/hosp/services.csv.gz
Size: 8.17 MB
----------------------------------------
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/hosp/d_hcpcs.csv.gz
Size: 0.41 MB
----------------------------------------
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/hosp/poe_detail.csv.gz
Size: 52.71 MB
----------------------------------------
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/hosp/emar.csv.gz
Size: 773.72 MB
----------------------------------------
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/hosp/hcpcsevents.csv.gz
Size: 2.06 MB
----------------------------------------
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/hosp/provider.csv.gz
Size: 0.12 MB
----------------------------------------
/content/dri

**Cấu trúc các bảng**
1. `chartevents`: Tất cả các thông tin được ghi lại trên biểu đồ để theo dõi bệnh nhân bởi điều dưỡng hoặc máy móc
2. `procedureevents`: Thông tin về các thủ thuật lâm sàng trong quá trình hồi sức
3. `ingredientevents`: Bóc tách chi tiết lượng chính xác của từng thành phần hoạt chất có trong hỗn hợp dược ghi trong `inputevents`
4. `caregiver`: Lưu trữ thông tin về những người trực tiếp thực hiện các thao tác chăm sóc hoặc ghi nhận dữ liệu
5. `inputevents`: Theo dõi cân bằng dịch và thuốc truyền tĩnh mạch vào đi với bảng `ingredientevents`
6. `d_items`: Đây là bảng tra cứu (Dictionary). Nó giải nghĩa cho các itemid xuất hiện trong chartevents, inputevents, v.v. Nếu bạn muốn tìm xem mã nào là "Nhịp tim", bạn phải tìm trong bảng này.
7. `datetimeevents`: Mốc thời gian quan trọng khác trong hồ sơ bệnh nhân mà không thuộc dạng đo lường định kỳ, ví dụ như thời gian dự kiến rút ống nội khí quản
8. `outputevents`: Theo dõi cân bằng dịch và thuốc truyền tĩnh mạch ra
9. `icustays:` Quản lý hành chính cho mỗi lần bệnh nhân nằm tại ICU

**Cách mà các bảng kết nối với nhau (qua các mã định danh)**

- `stay_id:` Biết thông tin đó thuộc về đợt nằm ICU nào của bệnh nhân nào

- `itemid:` Tra cứu trong bảng `d_items` để biết con số đó là gì (ví dụ: itemid là 220001 tương ứng với "Danh sách vấn đề")

- `caregiver_id:` Một mã số đại diện cho nhân viên y tế (bác sĩ, điều dưỡng) đã nhập dữ liệu đó vào hệ thống. Mã này giúp liên kết các sự kiện với bảng `caregiver` để biết ai là người ghi nhận thông tin, nhưng danh tính thật của họ đã được bảo mật bằng các con số ngẫu nhiên




In [ ]:
read_file(hosp)

File: /content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/hosp/prescriptions.csv.gz
 - Kích thước: 20292611 dòng x 21 cột
 - Các cột: ['subject_id', 'hadm_id', 'pharmacy_id', 'poe_id', 'poe_seq', 'order_provider_id', 'starttime', 'stoptime', 'drug_type', 'drug', 'formulary_drug_cd', 'gsn', 'ndc', 'prod_strength', 'form_rx', 'dose_val_rx', 'dose_unit_rx', 'form_val_disp', 'form_unit_disp', 'doses_per_24_hrs', 'route']

------------------------------
File: /content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/hosp/services.csv.gz
 - Kích thước: 593071 dòng x 5 cột
 - Các cột: ['subject_id', 'hadm_id', 'transfertime', 'prev_service', 'curr_service']

------------------------------
File: /content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/hosp/d_hcpcs.csv.gz
 - Kích thước: 89208 dòng x 4 cột
 - Các cột: ['code', 'category', 'long_description', 'short_description']

------------------------------
File: /content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1

### **ICU**

In [ ]:
# ICU
icu = check_file('/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/icu/*.csv.gz')

Số file: 9
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/icu/chartevents.csv.gz
Size: 3340.14 MB
----------------------------------------
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/icu/procedureevents.csv.gz
Size: 22.98 MB
----------------------------------------
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/icu/ingredientevents.csv.gz
Size: 297.21 MB
----------------------------------------
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/icu/caregiver.csv.gz
Size: 0.04 MB
----------------------------------------
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/icu/inputevents.csv.gz
Size: 382.51 MB
----------------------------------------
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/icu/d_items.csv.gz
Size: 0.06 MB
----------------------------------------
/content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/icu/datetimeevents.csv.gz
Size: 60.54 MB
-----------------------------------

**Cấu trúc các bảng**
1. `chartevents`: Tất cả các thông tin được ghi lại trên biểu đồ để theo dõi bệnh nhân bởi điều dưỡng hoặc máy móc
2. `procedureevents`: Thông tin về các thủ thuật lâm sàng trong quá trình hồi sức
3. `ingredientevents`: Bóc tách chi tiết lượng chính xác của từng thành phần hoạt chất có trong hỗn hợp dược ghi trong `inputevents`
4. `caregiver`: Lưu trữ thông tin về những người trực tiếp thực hiện các thao tác chăm sóc hoặc ghi nhận dữ liệu
5. `inputevents`: Theo dõi cân bằng dịch và thuốc truyền tĩnh mạch vào đi với bảng `ingredientevents`
6. `d_items`: Đây là bảng tra cứu (Dictionary). Nó giải nghĩa cho các itemid xuất hiện trong chartevents, inputevents, v.v. Nếu bạn muốn tìm xem mã nào là "Nhịp tim", bạn phải tìm trong bảng này.
7. `datetimeevents`: Mốc thời gian quan trọng khác trong hồ sơ bệnh nhân mà không thuộc dạng đo lường định kỳ, ví dụ như thời gian dự kiến rút ống nội khí quản
8. `outputevents`: Theo dõi cân bằng dịch và thuốc truyền tĩnh mạch ra
9. `icustays:` Quản lý hành chính cho mỗi lần bệnh nhân nằm tại ICU

**Cách mà các bảng kết nối với nhau (qua các mã định danh)**

- `stay_id:` Biết thông tin đó thuộc về đợt nằm ICU nào của bệnh nhân nào

- `itemid:` Tra cứu trong bảng `d_items` để biết con số đó là gì (ví dụ: itemid là 220001 tương ứng với "Danh sách vấn đề")

- `caregiver_id:` Một mã số đại diện cho nhân viên y tế (bác sĩ, điều dưỡng) đã nhập dữ liệu đó vào hệ thống. Mã này giúp liên kết các sự kiện với bảng `caregiver` để biết ai là người ghi nhận thông tin, nhưng danh tính thật của họ đã được bảo mật bằng các con số ngẫu nhiên




In [ ]:
read_file(icu)

File: /content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/icu/chartevents.csv.gz
 - Kích thước: 432997491 dòng x 11 cột
 - Các cột: ['subject_id', 'hadm_id', 'stay_id', 'caregiver_id', 'charttime', 'storetime', 'itemid', 'value', 'valuenum', 'valueuom', 'warning']

------------------------------
File: /content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/icu/procedureevents.csv.gz
 - Kích thước: 808706 dòng x 22 cột
 - Các cột: ['subject_id', 'hadm_id', 'stay_id', 'caregiver_id', 'starttime', 'endtime', 'storetime', 'itemid', 'value', 'valueuom', 'location', 'locationcategory', 'orderid', 'linkorderid', 'ordercategoryname', 'ordercategorydescription', 'patientweight', 'isopenbag', 'continueinnextdept', 'statusdescription', 'originalamount', 'originalrate']

------------------------------
File: /content/drive/MyDrive/NCKH-DDU1231/physionet.org/mimiciv/3.1/icu/ingredientevents.csv.gz
 - Kích thước: 14253480 dòng x 17 cột
 - Các cột: ['subject_id', 'hadm_id', 'stay_id', 

#

# **2. Thực hiện việc kiểm định dữ liệu**

Dữ liệu mapping bao gồm có hai bảng sheet:
- `All:` Bảng này được join từ bao gồm 6 bảng nhằm lưu trữ toàn bộ các biến có thể được thu thập từ hệ thống thông tin bệnh viện, thực hiện gom từ 6 bảng
- `Selected_Item:` Bảng này ánh xạ biến nghiên cứu dùng để kết nối các biến lâm sàng tại bệnh viện và chia thành các nhóm lớn

In [ ]:
all = check_file('/content/drive/MyDrive/NCKH-DDU1231/Mapping/*.csv') # Kiểm tra file có trong folder mapping

Số file: 2
/content/drive/MyDrive/NCKH-DDU1231/Mapping/Selected_Item.csv
Size: 0.01 MB
----------------------------------------
/content/drive/MyDrive/NCKH-DDU1231/Mapping/All.csv
Size: 1.21 MB
----------------------------------------


In [ ]:
def check_duplicate(df):
  if df.duplicated().any():
    print("Có giá trị bị trùng lặp")
  else:
    print("Không có giá trị bị trùng lặp")

## **Kiểm tra bảng All trong file Mapping**

Các bảng có `itemid` làm khóa ngoại `d_labitems`, `d_icd_procedures`, `chartevents`,`procedureevents`, `datetimeevents`, `outputevents`

- `pharmacy`: `itemid` == `medication` tên dược phẩm hỗn hợp thuốc pha chế

In [ ]:
all = pd.read_csv('/content/drive/MyDrive/NCKH-DDU1231/Mapping/All.csv') # Đọc dữ liệu
all.head()

,Table,itemid,Description,Type,N,Example
0,chartevents,220001,Problem List,Category,583305,".Care Plan_Impaired Tissue Perfusion, .Care Pl..."
1,chartevents,220045,Heart Rate,Numeric,8752069,"Min = -241395.00, max = 10000000.00, mean = 88..."
2,chartevents,220046,Heart rate Alarm_High,Numeric,843381,"Min = 0.00, max = 180160.00, mean = 131.74, st..."
3,chartevents,220047,Heart Rate Alarm_Low,Numeric,843694,"Min = -55.00, max = 695434.00, mean = 60.90, s..."
4,chartevents,220048,Heart Rhythm,Category,7947289,"SR (Sinus Rhythm), ST (Sinus Tachycardia) , AF..."


In [ ]:
all.shape

(14770, 6)

**Mô tả đặc trưng**

- `Table:` Bảng lấy
- `itemid:` Lấy từ `d_item` mã bệnh v.v..
- `Description:` Lấy từ `d_item` mô tả các mã
- `Type:` Kiểu dữ liệu
- `N:` Các mẫu có trong bảng
- `Example:` Ví dụ cho từng dữ liệu trong bảng

In [ ]:
check_duplicate(all)

Không có giá trị bị trùng lặp


## **Kiểm tra bảng Selected_Item trong file Mapping**

In [ ]:
col_name = ['Nhóm', 'BVQ11', 'MIMICIV Table', 'ID/Column', 'Description', 'Example', 'Notes']

In [ ]:
selected_item = pd.read_csv('/content/drive/MyDrive/NCKH-DDU1231/Mapping/Selected_Item.csv', names=col_name, header=1) # Đọc dữ liệu
selected_item.head()

,Nhóm,BVQ11,MIMICIV Table,ID/Column,Description,Example,Notes
0,Bệnh nhân,Tuổi,patients,anchor_age,Age,NaN,NaN
1,Bệnh nhân,Giới tính,patients,gender,Gender,NaN,NaN
2,Bệnh nhân,Chiều cao,chartevents,226707,Height,NaN,NaN
3,Bệnh nhân,Cân nặng,chartevents,226512,Weight,NaN,NaN
4,Bệnh nhân,Bệnh Nội khoa,infection_ids,All,-,NaN,NaN


In [ ]:
selected_item.shape

(68, 7)

**Mô tả đặc trưng**

- `Nhóm:` Phân loại nhóm bệnh nhân
- `BVQ11:` Mô tả nhóm
- `MIMICIV Table:` Đối chiếu Bảng MIMIC
- `ID/Column:` Mã/Cột trong bảng đối chiếu
- `Description:` Mô tả

## **Kiểm tra bảng Selected_Item có thuộc tính nào itemid nào trong All**


In [ ]:
def find_item(df_find, pattern_list, col):
    # Tạo một danh sách các dataframe kết quả
    frames = []
    for p in pattern_list:
        # Lọc không phân biệt hoa thường
        matched = df_find[df_find[col].str.contains(p, case=False, na=False)]
        frames.append(matched)

    # Gộp tất cả lại, xóa trùng lặp và sắp xếp theo itemid
    result = pd.concat(frames).drop_duplicates().sort_values('itemid')
    return result

In [ ]:
# Tìm cả 'SpO2' và 'Oxygen saturation'
pattern = [
    r'SpO2|Oxygen saturation',               # Nhóm SpO2
    r'\bpH\b',                               # Nhóm pH (chỉ khớp đúng từ pH)
    r'FiO2|Inspired Oxygen|Fraction inspired',# Nhóm FiO2
    r'Troponin I',                           # Chức năng tim
    r'Albumin',                              # Chức năng gan (Albumin)
    r'Bilirubin.*Indirect',                  # Bilirubin gián tiếp (GT)
    r'Oxygen Therapy|O2 Therapy',            # Oxy liệu pháp
    r'Vancomycin|Ceftriaxone|Antibiotic'     # Kháng sinh (ví dụ một số loại phổ biến)
]

# Chạy hàm với danh sách pattern mới
df_verify = find_item(all, pattern, 'Description')

# Hiển thị kết quả để verify
df_verify

,Table,itemid,Description,Type,N,Example
32,chartevents,220274,PH (Venous),Numeric,90350,"Min = 0.00, max = 999999.00, mean = 239.78, st..."
54,chartevents,220734,PH (dipstick),Numeric,52255,"Min = 5.00, max = 9.00, mean = 6.01, std = 0.77"
2311,inputevents,220862,Albumin 25%,Numeric,26470,"Min = 0.22, max = 1000.00, mean = 73.12, std =..."
2312,inputevents,220864,Albumin 5%,Numeric,30906,"Min = 2.08, max = 1500.00, mean = 338.73, std ..."
110,chartevents,223830,PH (Arterial),Numeric,432459,"Min = 0.00, max = 999999.00, mean = 143.80, st..."
...,...,...,...,...,...,...
14517,prescriptions,vancomycin,vancomycin,Category,1,vancomycin
14518,prescriptions,vancomycin (bulk),vancomycin (bulk),Category,7,vancomycin (bulk)
14519,prescriptions,vancomycin CAPS,vancomycin CAPS,Category,1,vancomycin CAPS
14520,prescriptions,vancomycin capsule,vancomycin capsule,Category,2,vancomycin capsule


In [ ]:
df_verify.shape

(110, 6)